This script parses raw German vehicle registration CSVs (2021–2025) for the Kiel district (code 01002), covering vehicle type and propulsion type data. It normalizes inconsistent column names across years, merges both sources per year into a single row, and saves the combined output as one CSV file.

Developed with the assistance of [Claude (Anthropic)](https://claude.ai)

In [4]:
import csv
import re

INPUT_FOLDER = "../../data/RegistrationData/RawData/"
OUTPUT_FOLDER = "../../data/RegistrationData/"
YEARS = [2021, 2022, 2023, 2024, 2025]
SOURCES = [
    ("vehicle_type", "VT"),
    ("propulsion_type", "PT"),
]

# Metadata columns to skip
IGNORE = {"", "Land", "Regierungsbezirk", "Statistische Kennziffer und Zulassungsbezirk"}

# Normalize column names that were renamed across years
RENAMES = {
    # FahrzeugArt: gendering changes (2021/22 → from 2023)
    "FZ_Krafträder_darunter weibliche Halter":                          "FZ_Krafträder_darunter Halterinnen",
    "FZ_Personenkraftwagen_und zwar gewerbliche Halter":                "FZ_Personenkraftwagen_und zwar gewerbliche Halterinnen und Halter",
    "FZ_Personenkraftwagen_und zwar weibliche Halter":                  "FZ_Personenkraftwagen_und zwar Halterinnen",
    # FahrzeugArt: "darunter" → "davon" (2021 → from 2022)
    "FZ_Zugmaschinen_darunter Sattelzugmaschinen":                      "FZ_Zugmaschinen_davon Sattelzugmaschinen",
    "FZ_Zugmaschinen_darunter land-/forstwirtschaftliche Zugmaschinen": "FZ_Zugmaschinen_davon land-/forstwirtschaftliche Zugmaschinen",
    "FZ_Zugmaschinen_land-/forstwirtschaftliche Zugmaschinen beinhalten leichte Zug-maschinen": "FZ_Zugmaschinen_davon sonstige Zugmaschinen",
    # AntriebsArt: spelling variant in last column
    "AA_Darunter dieselangetriebene Pkw nach Emissionsgruppen_schadstoffred. mit Dieselantrieb insgesamt":
        "AA_Darunter dieselangetriebene Pkw nach Emissionsgruppen_schadstoffreduziert mit Dieselantrieb insgesamt",
}


def clean(s):
    """Normalize whitespace and remove hyphenation artifacts."""
    return re.sub(r'- ', '', re.sub(r'\s+', ' ', s)).strip()


def build_column_names(row0, row1, label):
    """Build prefixed column names from two header rows, forward-filling the top category."""
    # Forward-fill top-level category
    top = ""
    columns = []
    for t, b in zip(row0, row1):
        t, b = clean(t), clean(b)
        if t:
            top = t
        if top in IGNORE or b in IGNORE:
            columns.append("")
        elif not top and not b:
            columns.append("")
        elif not top:
            columns.append(f"{label}_{b}")
        elif not b or top == b:
            columns.append(f"{label}_{top}")
        else:
            columns.append(f"{label}_{top}_{b}")
    return columns


def load_kiel(file_prefix, label, year):
    """Load a single source file and return the row for Kiel (district code 01002)."""
    path = f"{INPUT_FOLDER}Data {year}/{file_prefix}_{year}.CSV"
    with open(path, encoding="latin-1", newline="") as f:
        rows = list(csv.reader(f, delimiter=";"))

    columns = build_column_names(rows[0], rows[1], label)

    for row in rows[2:]:
        if len(row) > 2 and row[2].strip().startswith("01002"):
            return {
                RENAMES.get(col, col): val
                for col, val in zip(columns, row)
                if col
            }


# Load all data
all_rows = []
for year in YEARS:
    row = {"Year": year}
    for file_prefix, label in SOURCES:
        row.update(load_kiel(file_prefix, label, year))
    all_rows.append(row)

# Collect all column names across years (preserving order)
all_columns = list(dict.fromkeys(k for row in all_rows for k in row))

output_path = f"{OUTPUT_FOLDER}kiel_registration_data.CSV"
with open(output_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=all_columns, delimiter=",", extrasaction="ignore")
    writer.writeheader()
    writer.writerows(all_rows)

2021: loaded
2022: loaded
2023: loaded
2024: loaded
2025: loaded

Done! 69 columns, 5 rows saved.
